# Project 4 — Movie Recommendation System (Colab-first workflow)

**The problem:** given a movie someone liked, how do we suggest
similar ones — without hand-writing rules like
*"if you liked Interstellar, suggest The Martian"*?

We'll build a **content-based recommender**: it compares the text
*descriptions* of movies to each other, using the same TF-IDF
technique from Project 1 — but this time to measure **similarity**
instead of classifying Positive/Negative.

## Step 1: Install dependencies

In [ ]:
!pip install -q scikit-learn pandas

## Step 2: Create the dataset

A small table of 12 movies, each with a short description made of
genre/theme/setting keywords. These descriptions are what the system
actually compares — not posters, cast, or reviews.

In [ ]:
import pandas as pd

movies = pd.DataFrame({
    "title": [
        "Interstellar", "Inception", "The Martian", "Arrival",
        "The Matrix", "Avatar", "Titanic", "The Notebook",
        "Avengers: Endgame", "Iron Man", "Jurassic Park", "The Dark Knight"
    ],
    "description": [
        "space science fiction astronauts future adventure",
        "science fiction dreams technology thriller mind bending",
        "space science fiction astronaut survival mars adventure",
        "science fiction aliens language space mystery",
        "science fiction technology artificial intelligence action",
        "science fiction space aliens adventure fantasy",
        "romance drama ship ocean historical tragedy",
        "romance relationship love drama emotional",
        "superhero action marvel time travel adventure",
        "superhero action technology marvel engineering",
        "dinosaurs science adventure action island",
        "superhero action crime batman thriller"
    ]
})

movies

## Step 3: Convert descriptions into numbers

Same `TfidfVectorizer` as Project 1. Every unique word across all 12
descriptions becomes one dimension of a vector — but this time we
won't classify these vectors, we'll compare them to each other.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

movie_vectorizer = TfidfVectorizer(stop_words="english")

movie_matrix = movie_vectorizer.fit_transform(
    movies["description"]
)

print("Movie matrix shape:", movie_matrix.shape)

## Step 4: Measure similarity between every pair of movies

`cosine_similarity()` compares every movie's vector against every
other movie's vector, producing a score from 0 (unrelated) to 1
(identical) for each pair. With 12 movies, this gives a 12x12 table —
row `i`, column `j` is how similar movie `i` is to movie `j`.

A movie compared with itself always scores 1.0 — we'll need to
exclude that when recommending, which the next function does.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(movie_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

## Step 5: Build the recommendation function

Step by step, this function:
1. Confirms the movie title exists in our dataset
2. Finds that movie's row in the similarity matrix
3. Sorts all other movies by similarity score, highest first
4. Removes the movie's comparison with itself (always 1.0)
5. Returns the top few matches

In [ ]:
def recommend_movies(movie_title, number_of_recommendations=5):
    if movie_title not in movies["title"].values:
        return f"Movie '{movie_title}' was not found."

    movie_index = movies.index[
        movies["title"] == movie_title
    ][0]

    similarity_scores = list(
        enumerate(similarity_matrix[movie_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = [
        item for item in similarity_scores
        if item[0] != movie_index
    ]

    recommendations = []

    for index, score in similarity_scores[:number_of_recommendations]:
        recommendations.append({
            "movie": movies.iloc[index]["title"],
            "similarity": round(score, 3)
        })

    return pd.DataFrame(recommendations)

## Step 6: Test it

In [ ]:
recommend_movies("Interstellar")

In [ ]:
recommend_movies("Iron Man")

In [ ]:
recommend_movies("Titanic")

### What did the AI actually do?

We never hard-coded *"if you liked Interstellar, recommend The
Martian."* The system compared numeric representations of movie
descriptions and found the closest matches mathematically. Real
platforms (Netflix, Spotify) combine this with watch history,
ratings, and other signals — this is the simplified core mechanism
underneath all of them.

## Step 7: Now turn this into files

Same pattern as Projects 1-3 — write the working logic out to
`data.py`, `model.py`, `app.py` (Streamlit version), plus
`requirements.txt`.

In [ ]:
%%writefile data.py
"""
Movie dataset for Project 4.
12 movies, each with a short keyword-style description covering
genre, themes, and setting — this is what the system compares.
"""

import pandas as pd


def build_movies_dataframe() -> pd.DataFrame:
    return pd.DataFrame({
        "title": [
            "Interstellar", "Inception", "The Martian", "Arrival",
            "The Matrix", "Avatar", "Titanic", "The Notebook",
            "Avengers: Endgame", "Iron Man", "Jurassic Park", "The Dark Knight",
        ],
        "description": [
            "space science fiction astronauts future adventure",
            "science fiction dreams technology thriller mind bending",
            "space science fiction astronaut survival mars adventure",
            "science fiction aliens language space mystery",
            "science fiction technology artificial intelligence action",
            "science fiction space aliens adventure fantasy",
            "romance drama ship ocean historical tragedy",
            "romance relationship love drama emotional",
            "superhero action marvel time travel adventure",
            "superhero action technology marvel engineering",
            "dinosaurs science adventure action island",
            "superhero action crime batman thriller",
        ],
    })


In [ ]:
%%writefile model.py
"""
Content-based recommendation logic for Project 4.
Same technique as Project 1 (TF-IDF), but used to measure
*similarity* between movies instead of classifying text.
"""

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from data import build_movies_dataframe


def build_recommender():
    """
    Builds the TF-IDF matrix and the movie-to-movie similarity matrix.
    Returns (movies_df, similarity_matrix).
    """
    movies = build_movies_dataframe()

    vectorizer = TfidfVectorizer(stop_words="english")
    movie_matrix = vectorizer.fit_transform(movies["description"])

    similarity_matrix = cosine_similarity(movie_matrix)

    return movies, similarity_matrix


def recommend_movies(movies: pd.DataFrame, similarity_matrix, movie_title: str, number_of_recommendations: int = 5):
    """Returns a DataFrame of the most similar movies to movie_title."""
    if movie_title not in movies["title"].values:
        return None

    movie_index = movies.index[movies["title"] == movie_title][0]

    similarity_scores = list(enumerate(similarity_matrix[movie_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = [item for item in similarity_scores if item[0] != movie_index]

    recommendations = []
    for index, score in similarity_scores[:number_of_recommendations]:
        recommendations.append({
            "movie": movies.iloc[index]["title"],
            "similarity": round(float(score), 3),
        })

    return pd.DataFrame(recommendations)


In [ ]:
%%writefile app.py
"""
AI Playground — Project 4: Movie Recommendation System
Streamlit app version.

Run locally:
    streamlit run app.py
"""

import streamlit as st

from model import build_recommender, recommend_movies

st.set_page_config(
    page_title="Movie Recommender",
    page_icon="🎬",
    layout="centered",
)


@st.cache_resource(show_spinner="Building the similarity matrix...")
def get_recommender():
    return build_recommender()


movies, similarity_matrix = get_recommender()

st.title("🎬 Movie Recommendation System")
st.caption("Content-based filtering: TF-IDF + cosine similarity, no user data needed.")

with st.expander("How this works"):
    st.markdown(
        """
        This is a **content-based** recommender — it suggests movies
        purely by comparing text descriptions, not by studying what
        other users watched.

        1. Every movie's description is converted to numbers with
           **TF-IDF** (same technique as Project 1).
        2. **Cosine similarity** compares every movie's numbers
           against every other movie's, producing a score from 0
           (unrelated) to 1 (identical) for every pair.
        3. To recommend, we just look up the row for your chosen
           movie and return the highest-scoring others.

        Nothing here was hard-coded like *"if you liked X, suggest Y"*
        — every suggestion comes purely from shared keywords in the
        descriptions.
        """
    )

st.subheader("Pick a movie")
selected_movie = st.selectbox("You liked:", movies["title"].tolist())
num_recs = st.slider("Number of recommendations", 1, 8, 5)

if st.button("Get Recommendations", type="primary"):
    result = recommend_movies(movies, similarity_matrix, selected_movie, num_recs)
    st.subheader(f"Because you liked {selected_movie}")
    st.dataframe(result, use_container_width=True, hide_index=True)

with st.expander("See the full movie catalog"):
    st.dataframe(movies, use_container_width=True, hide_index=True)

st.caption("Part of the AI Playground: 4 Real-World AI Projects series.")


In [ ]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn


## Step 8: Launch the Streamlit app from Colab (optional)

In [ ]:
!pip install -q streamlit
!wget -q -O - ipv4.icanhazip.com
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501